In [3]:
from os import environ
from dotenv import load_dotenv
# from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain.chains import ConversationChain, LLMChain
from langchain.memory import ConversationBufferMemory
from langchain.agents import AgentExecutor, create_tool_calling_agent, initialize_agent, AgentType, load_tools

In [4]:
# Read API KEY
load_dotenv()
api_key_groq = environ['API_GROG']

In [18]:
# Load LLM
llm = ChatGroq(
    model="llama3-70b-8192",
    temperature=0.5,
    max_tokens=50,
    api_key=api_key_groq
)

In [6]:
# Create tools
@tool
def investing(capital: int, tenure: float, dividend_yield: float) -> int:
    """Calculate the final value of investment given the capital, interest rate per annum, and tenure.
    If the dividend yield is in percentage (%), than divide it with 100 to convert it to be decimal."""
    invest = capital * (1 + dividend_yield)**tenure
    return invest

@tool
def calculate_dy(current_price: int, dividend: int) -> float:
    """Calculate dividend yield based on the dividend and and current price"""
    dividend_yield = round(dividend/current_price, 3)
    return dividend_yield

@tool
def entry_criteria(dividend_yield: float, current_price: int, yearly_payout: bool) -> str:
    """Decide whether to buy the stock (entry) based on the dividend yield, current price, and yearly payout"""
    if yearly_payout == True:
        if dividend_yield >= 0.04 or current_price <= 1000:
            decision = "buy the stock"
        else:
            decision = "do not buy the stock"
    else:
        if dividend_yield >= 0.6 and current_price <= 3000:
            decision = "buy the stock"
        else:
            decision = "do not buy the stock"
    
    return decision


In [7]:
# Create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer not more than 20 words"),
        MessagesPlaceholder("history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

# Regular agent

In [8]:
tools = [investing, calculate_dy, entry_criteria]
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [9]:
response = agent_executor.invoke({"input": "Hey, what's up?"})
response



> Entering new AgentExecutor chain...
Not much! Just hanging out. How about you?

> Finished chain.


{'input': "Hey, what's up?",
 'output': 'Not much! Just hanging out. How about you?'}

In [10]:
response = agent_executor.invoke({"input": "Define finance in less than 10 words."})
response



> Entering new AgentExecutor chain...
Management of money and investments for individuals and organizations.

> Finished chain.


{'input': 'Define finance in less than 10 words.',
 'output': 'Management of money and investments for individuals and organizations.'}

In [22]:
response = agent_executor.invoke(
    {"input": "if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?"}
)
response



> Entering new AgentExecutor chain...

Invoking: `calculate_dy` with `{'current_price': 270, 'dividend': 30}`


0.111You got a dividend yield of 0.111, which is equivalent to 11.1%.

> Finished chain.


{'input': 'if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?',
 'output': 'You got a dividend yield of 0.111, which is equivalent to 11.1%.'}

In [23]:
response = agent_executor.invoke(
    {"input": "if I buy a stock with the dividend of 30 when the price is 270. The stock pays dividend every year. Should I buy the stock?"}
)
response



> Entering new AgentExecutor chain...

Invoking: `calculate_dy` with `{'current_price': 270, 'dividend': 30}`


0.111
Invoking: `entry_criteria` with `{'current_price': 270, 'dividend_yield': 0.111, 'yearly_payout': True}`


buy the stockYou should buy the stock.

> Finished chain.


{'input': 'if I buy a stock with the dividend of 30 when the price is 270. The stock pays dividend every year. Should I buy the stock?',
 'output': 'You should buy the stock.'}

In [49]:
response = agent_executor.invoke({"input": "The stock pays dividend every year. Should I buy the stock?"})
response



> Entering new AgentExecutor chain...


> Finished chain.


{'input': 'The stock pays dividend every year. Should I buy the stock?',
 'output': ''}

# Zero-shot agent

## External tools

In [28]:
# Zero-shot agent with external tools
tools = load_tools(["llm-math"], llm=llm)
agent_external = initialize_agent(tools , llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)

C:\Users\USER\AppData\Local\Temp\ipykernel_35028\3555256760.py:3: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent_external = initialize_agent(tools , llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)


In [29]:
response = agent_external.run("what is 6 * 11?")

C:\Users\USER\AppData\Local\Temp\ipykernel_35028\980488979.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent_external.run("what is 6 * 11?")




> Entering new AgentExecutor chain...
Thought: I need to calculate the product of 6 and 11.

Action: Calculator
Action Input: 6 * 11
Observation: Answer: 66
Thought:I now know the final answer
Final Answer: 66

> Finished chain.


In [30]:
response = agent_external.run("(4 + 5)/3*2 is ")



> Entering new AgentExecutor chain...
Thought: I need to evaluate the expression (4 + 5)/3*2.

Action: Calculator
Action Input: (4 + 5)/3*2
Observation: Answer: 6.0
Thought:I now know the final answer
Final Answer: 6.0

> Finished chain.


## Custom tools

In [31]:
# Create zero-shot agent
agent_zeroshot = initialize_agent(
    tools=[investing, calculate_dy, entry_criteria],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
)

In [32]:
response = agent_zeroshot.run("if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?")
response



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "calculate_dy",
  "action_input": {
    "current_price": 270,
    "dividend": 30
  }
}
```

Observation: 0.111
Thought:The dividend yield is 0.111 or 11.1%.

> Finished chain.


'The dividend yield is 0.111 or 11.1%.'

In [33]:
response = agent_zeroshot.run("if I buy a stock with the dividend of 30 when the price is 270. The stock pays dividend every year. Should I buy the stock?")
response



> Entering new AgentExecutor chain...
Question: Should I buy the stock?

Thought: To determine if I should buy the stock, I need to calculate the dividend yield and evaluate it based on the current price and yearly payout.

Action:
```
{
  "action": "calculate_dy

> Finished chain.


'Question: Should I buy the stock?\n\nThought: To determine if I should buy the stock, I need to calculate the dividend yield and evaluate it based on the current price and yearly payout.\n\nAction:\n```\n{\n  "action": "calculate_dy'

# Add memory

In [35]:
# Create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Always call me 'Boss' and answer not more than 20 words"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

In [36]:
# Bind tools
tools = [investing, calculate_dy, entry_criteria]
llm_tools = llm.bind_tools(tools)

In [37]:
from langchain.agents.format_scratchpad.openai_tools import format_to_openai_tool_messages
from langchain.agents.output_parsers.openai_tools import OpenAIToolsAgentOutputParser

In [38]:
# Create agent
agent = (
    {
        "input": lambda x: x["input"],
        "agent_scratchpad": lambda x: format_to_openai_tool_messages(x["intermediate_steps"]),
        "history": lambda x: x["history"],
    }
    | prompt
    | llm_tools
    | OpenAIToolsAgentOutputParser()
)

In [39]:
# Create agent
agent_memory = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [40]:
# Create memory
memory = ConversationBufferMemory(llm=llm, memory_key="history")
# Prepopulate the memory with chat history
history_input = ["if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?"]
history_output = ["The dividend yield is 11.1%."]
if history_input is not None:
    for inp, outp in zip(history_input, history_output):
        memory.save_context({"input": inp}, {"output": outp})

C:\Users\USER\AppData\Local\Temp\ipykernel_35028\4235524173.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(llm=llm, memory_key="history")


In [41]:
response = agent_memory.invoke(
    {
        "input": "If the stock pays dividend yearly, should I buy the stock?",
        "history": memory.chat_memory.messages,
    }
)
response



> Entering new AgentExecutor chain...


> Finished chain.


{'input': 'If the stock pays dividend yearly, should I buy the stock?',
 'history': [HumanMessage(content='if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The dividend yield is 11.1%.', additional_kwargs={}, response_metadata={})],
 'output': ''}

# Add memory (2)
Add the memory to prompt

In [42]:
# Create prompt template
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Always call me 'Boss', Answer not more than 20 words"),
        MessagesPlaceholder("history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

In [43]:
tools = [investing, calculate_dy, entry_criteria]
agent = create_tool_calling_agent(llm, tools, prompt)
agent_memory_2 = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [44]:
# Create memory
memory = ConversationBufferMemory(llm=llm, memory_key="history")
# Prepopulate the memory with chat history
history_input = ["if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?"]
history_output = ["The dividend yield is 11.1%."]
if history_input is not None:
    for inp, outp in zip(history_input, history_output):
        memory.save_context({"input": inp}, {"output": outp})

In [45]:
question = "The stock pays dividend every year. Should I buy the stock?"
response = agent_memory_2.invoke(
    {
        "input": question,
        "history": memory.chat_memory.messages,
    }
)
response



> Entering new AgentExecutor chain...


> Finished chain.


{'input': 'The stock pays dividend every year. Should I buy the stock?',
 'history': [HumanMessage(content='if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The dividend yield is 11.1%.', additional_kwargs={}, response_metadata={})],
 'output': ''}

In [46]:
question = "If i invest 1000 it for 3 years, with the mentioned dividend yield, how much will I get?"
response = agent_memory_2.invoke(
    {
        "input": question,
        "history": memory.chat_memory.messages,
    }
)
response



> Entering new AgentExecutor chain...

Invoking: `investing` with `{'capital': 1000, 'dividend_yield': 0.111, 'tenure': 3}`


1371.330631Boss, you'll get approximately 1371.33 after 3 years.

> Finished chain.


{'input': 'If i invest 1000 it for 3 years, with the mentioned dividend yield, how much will I get?',
 'history': [HumanMessage(content='if I buy a stock with the dividend of 30 when the price is 270, how much is the dividend yield?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='The dividend yield is 11.1%.', additional_kwargs={}, response_metadata={})],
 'output': "Boss, you'll get approximately 1371.33 after 3 years."}